In [0]:
### Importando bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np



In [0]:
df_bureau_EDA_01 = spark.read.parquet(
    '/Volumes/hackathon_2025/default/source/base_score_bureau_movel/'
)
display(df_bureau_EDA_01)

**Significados dos books das tabelas **:

**Safra** (First Payment Default - Inadimplência na Primeira Parcela) é o indicador de "morte súbita". É o sinal mais precoce de que algo está errado na concessão.

**SCORE_01 ou SCORE_02** separa os bons dos maus pagadores , representa o Targuet 0 - adimplente 1- inadimplente

**FLAG_INSTALACAO** geralmente indica se o produto ou serviço foi efetivamente ativado/instalado após a aprovação.

** PROD e flag_mig2 ** permitem entender o mix de carteira.

O **FPD** (First Payment Default - Inadimplência na Primeira Parcela) é o indicador de "morte súbita". É o sinal mais precoce de que algo está errado na concessão.

**NUM_CPF** idenfica o cliente na base 



# Entendimento dos dados e do target

In [0]:
# verificando o tipo dos dados
df_bureau_EDA_01 = spark.read.parquet(
    '/Volumes/hackathon_2025/default/source/base_score_bureau_movel/') 
df_bureau_EDA_01.dtypes

In [0]:
#listando as colunas
list(df_bureau_EDA_01.columns)

In [0]:
# quantas linhas tem o dataframe
num_rows = df_bureau_EDA_01.count() 
num_cols = len(df_bureau_EDA_01.columns)
print(f"Rows: {num_rows}, Columns: {num_cols}")

In [0]:
# quantidade de safra que tem na base de dados
df_bureau_EDA_01.select('SAFRA').distinct().count()

In [0]:
# quais são as safras que tem na base de dados
df_bureau_EDA_01.select('SAFRA').distinct().sort('SAFRA').show()

In [0]:
# quantidade de CPF que tem na base de dados
total_cpfs = df_bureau_EDA_01.select('NUM_CPF').distinct().count()

print(f'Total de CPFs: {total_cpfs:,}')

In [0]:
# quantos CPFS aparecem em mais de uma safra (duplicados)
dup_count = df_bureau_EDA_01.groupBy('NUM_CPF').count().filter('count > 1').count() 

print(f'CPFs duplicados: {dup_count:,}')
#

In [0]:
percentual_cpfs_repetidos = dup_count / total_cpfs * 100
print(f"Percentual de cpfs repetidos nas safras: {percentual_cpfs_repetidos:.2f}%" )

In [0]:

# Estatísticas descritivas dos scores
df.select("SCORE_01", "SCORE_02").summary().display()


In [0]:
# Distribuição de CPFs por safra
cpf_por_safra = (
    df
    .groupBy("SAFRA")
    .agg(
        F.countDistinct("NUM_CPF").alias("qtd_cpfs")
    
    )
    .orderBy("SAFRA")
)

display(cpf_por_safra)


In [0]:
# Evolução percentual (crescimento ou queda)
from pyspark.sql.window import Window


window_safra = Window.orderBy("SAFRA")

cpf_por_safra = cpf_por_safra.withColumn(
    "var_pct_cpfs",
    (
        F.col("qtd_cpfs") - F.lag("qtd_cpfs").over(window_safra)
    ) / F.lag("qtd_cpfs").over(window_safra)
)

display(cpf_por_safra)


In [0]:
# Recorrência de CPFs entre safras
cpf_multisafra = (
    df
    .groupBy("NUM_CPF")
    .agg(
        F.countDistinct("SAFRA").alias("qtd_safras")
    )
)
display(cpf_multisafra)


In [0]:
distribuicao_multisafra = (
    cpf_multisafra
    .groupBy("qtd_safras")
    .agg(
        F.count("NUM_CPF").alias("qtd_cpfs")
    )
    .orderBy("qtd_safras")
)

display(distribuicao_multisafra)


**Interpretação :**

- a volumetria de cpfs que aparecem uma única vez nas safras é em torno de 98% , com isso a base não se comporta como o históricos de clientes , mas indica análises pontuais de perfil e risco.




In [0]:
# FPD por safra e nível de inadimplência
df_fpd = (
    df
    .groupBy("SAFRA", "FPD")
    .agg(
        F.countDistinct("NUM_CPF").alias("qtd_cpfs")
    )
    .orderBy("SAFRA", "FPD")
)
display(df_fpd)

# Analise de nulos da base 

In [0]:
# quantidade de nulos na base de dados

from pyspark.sql import functions as F

def contar_nulos(df):
    return df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

nulos_df = contar_nulos(bureau_EDA_01)


display(nulos_df)


In [0]:
# quantidades de valores nulos da variável score por FPD 
df_bureau_EDA_01.filter(F.col("SCORE_01").isNull() | F.col("SCORE_02").isNull()) \
    .groupBy("FPD").count().show()